# Arricchimento Titoli via Cellar SPARQL

Questo notebook recupera i titoli degli atti normativi del grafo focale tramite l'**API SPARQL ufficiale di EUR-Lex (Cellar)**.

## Perché questo step è necessario

Il dataset EurLex non include i titoli degli atti. I titoli sono fondamentali per la classificazione nei livelli normativi perché contengono informazioni strutturali esplicite non presenti nei metadati:

| Titolo | Layer |
|---|---|
| `Regulation of the European Parliament and of the Council` | **G2** |
| `Commission Delegated Regulation` | **G3** |
| `Commission Implementing Regulation` | **G3** |
| `Guidelines of the European Banking Authority` | **G4** |
| `Judgment of the Court of Justice` | **G5** |

## Parametri
- **API**: endpoint SPARQL Cellar
- **Delay**: 0.5s tra richieste
- **Checkpoint**: ogni 50 nodi
- **Input**: `data/output/golden_power/gephi_nodes_focal.csv`
- **Output**: `data/output/golden_power/gephi_nodes_focal_titled.csv`

In [1]:
import pandas as pd
import requests
import time
import os
import sys

sys.path.append('..')
from config_golden_power import MATERIA_NAME

output_path     = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file      = os.path.join(output_path, 'gephi_nodes_focal.csv')
output_file     = os.path.join(output_path, 'gephi_nodes_focal_titled.csv')
checkpoint_file = os.path.join(output_path, 'titles_checkpoint.csv')

SPARQL_ENDPOINT  = 'https://publications.europa.eu/webapi/rdf/sparql'
DELAY_SECONDS    = 0.5
CHECKPOINT_EVERY = 50
TIMEOUT          = 15

print(f"Input:      {input_file}")
print(f"Output:     {output_file}")
print(f"Checkpoint: {checkpoint_file}")

Input:      ..\data\output\golden_power\gephi_nodes_focal.csv
Output:     ..\data\output\golden_power\gephi_nodes_focal_titled.csv
Checkpoint: ..\data\output\golden_power\titles_checkpoint.csv


## 1. Caricamento Nodi e Gestione Checkpoint

In [2]:
nodes = pd.read_csv(input_file)
print(f"Nodi totali nel grafo focale: {len(nodes)}")

if os.path.exists(checkpoint_file):
    checkpoint   = pd.read_csv(checkpoint_file)
    already_done = set(checkpoint['Id'])
    print(f"Checkpoint trovato: {len(already_done)} nodi gia processati")
    print(f"Nodi rimanenti:     {len(nodes) - len(already_done)}")
else:
    checkpoint   = pd.DataFrame(columns=['Id', 'Label', 'title', 'title_status'])
    already_done = set()
    print("Nessun checkpoint trovato, si parte da zero")

nodes_todo = nodes[~nodes['Id'].isin(already_done)].copy()
print(f"Da processare ora: {len(nodes_todo)}")

Nodi totali nel grafo focale: 4904
Checkpoint trovato: 2766 nodi gia processati
Nodi rimanenti:     2138
Da processare ora: 2432


## 2. Funzione SPARQL

In [3]:
def get_title_from_cellar(celex, timeout=TIMEOUT):
    """
    Recupera il titolo inglese di un atto dall'API SPARQL di Cellar.
    Restituisce (title, status).
    """
    if pd.isna(celex) or celex == '':
        return None, 'not_found'

    query = f"""
    PREFIX cdm: <http://publications.europa.eu/ontology/cdm#>
    SELECT ?title WHERE {{
      ?work cdm:resource_legal_id_celex "{celex}"^^<http://www.w3.org/2001/XMLSchema#string> .
      ?expr cdm:expression_belongs_to_work ?work .
      ?expr cdm:expression_title ?title .
      FILTER(lang(?title) = "en")
    }}
    LIMIT 1
    """

    try:
        response = requests.post(
            SPARQL_ENDPOINT,
            data={'query': query, 'format': 'application/sparql-results+json'},
            headers={
                'Accept': 'application/sparql-results+json',
                'User-Agent': 'Mozilla/5.0 (academic research)',
            },
            timeout=timeout,
        )

        if response.status_code != 200:
            return None, f'error_{response.status_code}'

        bindings = response.json()['results']['bindings']
        if not bindings:
            return None, 'not_found'

        return bindings[0]['title']['value'], 'ok'

    except requests.exceptions.Timeout:
        return None, 'timeout'
    except Exception:
        return None, 'error'


# Test
print("Test su 32019R0452...")
title, status = get_title_from_cellar('32019R0452')
print(f"  Status: {status}")
print(f"  Titolo: {title}")

Test su 32019R0452...
  Status: ok
  Titolo: Regulation (EU) 2019/452 of the European Parliament and of the Council of 19 March 2019 establishing a framework for the screening of foreign direct investments into the Union


## 3. Fetch con Checkpoint

Con 2.432 nodi e 0.5s di delay il tempo stimato e circa **20 minuti**. Se viene interrotto, riesegui questa cella: ripartira dal checkpoint automaticamente.

In [4]:
results = []
errors  = []
total   = len(nodes_todo)

print(f"Inizio fetch: {total} nodi")
print(f"Tempo stimato: ~{total * DELAY_SECONDS / 60:.0f} minuti\n")

for i, (_, row) in enumerate(nodes_todo.iterrows()):
    celex   = row['Label']
    node_id = row['Id']

    title, status = get_title_from_cellar(celex)

    results.append({
        'Id':           node_id,
        'Label':        celex,
        'title':        title,
        'title_status': status,
    })

    if status != 'ok':
        errors.append((celex, status))

    if (i + 1) % 10 == 0 or (i + 1) == total:
        pct      = (i + 1) / total * 100
        ok_count = sum(1 for r in results if r['title_status'] == 'ok')
        print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {ok_count}  errori: {len(errors)}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        batch              = pd.DataFrame(results)
        checkpoint_updated = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
        checkpoint_updated.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint salvato ({len(checkpoint_updated)} nodi totali)")

    time.sleep(DELAY_SECONDS)

# Checkpoint finale
batch            = pd.DataFrame(results)
checkpoint_final = pd.concat([checkpoint, batch]).drop_duplicates(subset=['Id'])
checkpoint_final.to_csv(checkpoint_file, index=False)

print(f"\nFetch completato.")
print(f"  OK:        {(checkpoint_final['title_status'] == 'ok').sum()}")
print(f"  Not found: {(checkpoint_final['title_status'] == 'not_found').sum()}")
print(f"  Errori:    {(~checkpoint_final['title_status'].isin(['ok','not_found'])).sum()}")

Inizio fetch: 2432 nodi
Tempo stimato: ~20 minuti

  [  10/2432]   0.4%  ok: 7  errori: 3
  [  20/2432]   0.8%  ok: 7  errori: 13
  [  30/2432]   1.2%  ok: 7  errori: 23
  [  40/2432]   1.6%  ok: 12  errori: 28
  [  50/2432]   2.1%  ok: 22  errori: 28
  --> Checkpoint salvato (2816 nodi totali)
  [  60/2432]   2.5%  ok: 28  errori: 32
  [  70/2432]   2.9%  ok: 28  errori: 42
  [  80/2432]   3.3%  ok: 28  errori: 52
  [  90/2432]   3.7%  ok: 30  errori: 60
  [ 100/2432]   4.1%  ok: 39  errori: 61
  --> Checkpoint salvato (2866 nodi totali)
  [ 110/2432]   4.5%  ok: 49  errori: 61
  [ 120/2432]   4.9%  ok: 52  errori: 68
  [ 130/2432]   5.3%  ok: 52  errori: 78
  [ 140/2432]   5.8%  ok: 52  errori: 88
  [ 150/2432]   6.2%  ok: 61  errori: 89
  --> Checkpoint salvato (2916 nodi totali)
  [ 160/2432]   6.6%  ok: 71  errori: 89
  [ 170/2432]   7.0%  ok: 80  errori: 90
  [ 180/2432]   7.4%  ok: 80  errori: 100
  [ 190/2432]   7.8%  ok: 80  errori: 110
  [ 200/2432]   8.2%  ok: 83  errori: 11

## 4. Export

In [5]:
titles_df    = pd.read_csv(checkpoint_file)[['Id', 'title', 'title_status']]
nodes_titled = nodes.merge(titles_df, on='Id', how='left')
nodes_titled.to_csv(output_file, index=False)

print(f"File salvato: {output_file}")
print(f"  Nodi totali:  {len(nodes_titled)}")
print(f"  Con titolo:   {nodes_titled['title'].notna().sum()} ({nodes_titled['title'].notna().sum()/len(nodes_titled)*100:.1f}%)")
print(f"  Senza titolo: {nodes_titled['title'].isna().sum()}")
print()
print("Esempi titoli per LegalType:")
for tipo in nodes_titled['LegalType'].dropna().unique():
    subset  = nodes_titled[(nodes_titled['LegalType'] == tipo) & nodes_titled['title'].notna()]
    esempio = subset['title'].iloc[0] if len(subset) > 0 else 'N/A'
    print(f"  {tipo:<20} {str(esempio)[:80]}")

File salvato: ..\data\output\golden_power\gephi_nodes_focal_titled.csv
  Nodi totali:  4904
  Con titolo:   2530 (51.6%)
  Senza titolo: 2374

Esempi titoli per LegalType:
  Regulation           Regulation (EU) 2019/452 of the European Parliament and of the Council of 19 Mar
  Treaty               N/A
  Directive            COUNCIL DIRECTIVE 2008/114/EC of 8 December 2008 on the identification and desig
  Decision             POLITICAL AND SECURITY COMMITTEE DECISION BiH/8/2006 of 15 March 2006 amending D
  Recommendation       COMMISSION RECOMMENDATION of 22 December 2006 on safe and efficient in-vehicle i
  Legislative_Act      COUNCIL COMMON POSITION 2005/329/PESC of 25 April 2005 relating to the 2005 Revi
  Guidelines           Guideline (EU) 2024/2798 of the European Central Bank of 10 October 2024 amendin
  Complementary_Act    UN Regulation No 155 – Uniform provisions concerning the approval of vehicles wi
  Case_Law             Judgment of the Court of 17 September 2002.#Concor

## 5. Diagnostica

I nodi `not_found` sono tipicamente atti molto vecchi (anni '50-'60) o trattati con CELEX non standard. Per la classificazione LLM verranno gestiti tramite fallback sul preambolo (notebook `03b_enrich_preambles.ipynb`) o, in assenza anche di quello, sul titolo (notebook `04_semantic_classification.ipynb`).

In [6]:
print("Distribuzione status:")
print(nodes_titled['title_status'].value_counts().to_string())
print()

no_title = nodes_titled[nodes_titled['title'].isna()]
if len(no_title) > 0:
    print(f"Nodi senza titolo: {len(no_title)}")
    print("Per tipo:")
    print(no_title['LegalType'].value_counts().to_string())
    print()
    print("Per decade:")
    print(no_title['Decade'].value_counts().sort_index().to_string())
    print()
    print("Esempi CELEX senza titolo:")
    print(no_title['Label'].head(10).tolist())
else:
    print("Tutti i nodi hanno un titolo.")

Distribuzione status:
title_status
not_found    2371
ok           1803
ok_html       727
error_503       3

Nodi senza titolo: 2374
Per tipo:
LegalType
Decision             734
Treaty               485
Regulation           473
Directive            343
Legislative_Act      140
Case_Law             101
Recommendation        74
Complementary_Act     16
Guidelines             8

Per decade:
Decade
1950.0     52
1960.0     25
1970.0     75
1980.0    150
1990.0    550
2000.0    615
2010.0    874
2020.0     32

Esempi CELEX senza titolo:
['12016E063', '12016E065', '31964D0390', '31982D0861', '31985D0066', '31996D0546', '31996R0902', '31997D0486', '31997D0487', '31997D0524']


In [7]:
# ============================================================
# CELLA 6 — Retry not_found tramite HTML (get_html_by_celex_id)
# ============================================================
import eurlex
from bs4 import BeautifulSoup

def get_title_from_eurlex_html(celex):
    """
    Recupera il titolo da HTML EUR-Lex per i CELEX non trovati via SPARQL.
    Ricostruisce il titolo dai tag oj-doc-ti escludendo allegati.
    """
    try:
        html = eurlex.get_html_by_celex_id(celex)
        if not html:
            return None, 'not_found'

        soup  = BeautifulSoup(html, 'html.parser')
        parti = soup.find_all('p', class_='oj-doc-ti')

        if not parti:
            return None, 'not_found'

        titolo_parti = []
        for p in parti:
            testo = p.get_text(strip=True)
            if testo.startswith(('ANNEX', 'SCHEDULE', 'APPENDIX')):
                break
            titolo_parti.append(testo)

        if not titolo_parti:
            return None, 'not_found'

        return ' '.join(titolo_parti), 'ok_html'

    except Exception:
        return None, 'error_html'


# Carica il checkpoint attuale
checkpoint = pd.read_csv(checkpoint_file)
not_found  = checkpoint[checkpoint['title_status'] == 'not_found'].copy()

print(f"Nodi da ritentare via HTML: {len(not_found)}")
print(f"Tempo stimato: ~{len(not_found) * DELAY_SECONDS / 60:.0f} minuti\n")

retry_results = []
total = len(not_found)

for i, (_, row) in enumerate(not_found.iterrows()):
    celex   = row['Label']
    node_id = row['Id']

    title, status = get_title_from_eurlex_html(celex)
    retry_results.append({
        'Id':           node_id,
        'Label':        celex,
        'title':        title,
        'title_status': status,
    })

    if (i + 1) % 10 == 0 or (i + 1) == total:
        pct      = (i + 1) / total * 100
        ok_count = sum(1 for r in retry_results if r['title_status'] == 'ok_html')
        err      = sum(1 for r in retry_results if 'error' in r['title_status'])
        print(f"  [{i+1:>4}/{total}] {pct:5.1f}%  ok: {ok_count}  errori: {err}")

    if (i + 1) % CHECKPOINT_EVERY == 0:
        batch = pd.DataFrame(retry_results)
        # Aggiorna il checkpoint: sostituisci i not_found con i nuovi risultati
        checkpoint_updated = pd.concat([
            checkpoint[~checkpoint['Id'].isin(batch['Id'])],
            batch
        ]).drop_duplicates(subset=['Id'])
        checkpoint_updated.to_csv(checkpoint_file, index=False)
        print(f"  --> Checkpoint salvato")

    time.sleep(DELAY_SECONDS)

# Aggiorna checkpoint finale
batch = pd.DataFrame(retry_results)
checkpoint_final = pd.concat([
    checkpoint[~checkpoint['Id'].isin(batch['Id'])],
    batch
]).drop_duplicates(subset=['Id'])
checkpoint_final.to_csv(checkpoint_file, index=False)

print(f"\nRetry completato.")
print(f"  ok_html:    {(checkpoint_final['title_status'] == 'ok_html').sum()}")
print(f"  not_found:  {(checkpoint_final['title_status'] == 'not_found').sum()}")
print(f"  errori:     {checkpoint_final['title_status'].str.startswith('error').sum()}")

# Aggiorna il file finale
titles_df    = checkpoint_final[['Id', 'title', 'title_status']]
nodes_titled = nodes.merge(titles_df, on='Id', how='left')
nodes_titled.to_csv(output_file, index=False)

print(f"\nFile aggiornato: {output_file}")
print(f"  Con titolo:   {nodes_titled['title'].notna().sum()} ({nodes_titled['title'].notna().sum()/len(nodes_titled)*100:.1f}%)")
print(f"  Senza titolo: {nodes_titled['title'].isna().sum()}")

Nodi da ritentare via HTML: 2419
Tempo stimato: ~20 minuti

  [  10/2419]   0.4%  ok: 0  errori: 0
  [  20/2419]   0.8%  ok: 0  errori: 0
  [  30/2419]   1.2%  ok: 0  errori: 0
  [  40/2419]   1.7%  ok: 0  errori: 0
  [  50/2419]   2.1%  ok: 0  errori: 0
  --> Checkpoint salvato
  [  60/2419]   2.5%  ok: 0  errori: 0
  [  70/2419]   2.9%  ok: 0  errori: 0
  [  80/2419]   3.3%  ok: 0  errori: 0
  [  90/2419]   3.7%  ok: 0  errori: 0
  [ 100/2419]   4.1%  ok: 0  errori: 0
  --> Checkpoint salvato
  [ 110/2419]   4.5%  ok: 0  errori: 0
  [ 120/2419]   5.0%  ok: 0  errori: 0
  [ 130/2419]   5.4%  ok: 0  errori: 0
  [ 140/2419]   5.8%  ok: 0  errori: 0
  [ 150/2419]   6.2%  ok: 0  errori: 0
  --> Checkpoint salvato
  [ 160/2419]   6.6%  ok: 0  errori: 0
  [ 170/2419]   7.0%  ok: 0  errori: 0
  [ 180/2419]   7.4%  ok: 0  errori: 0
  [ 190/2419]   7.9%  ok: 0  errori: 0
  [ 200/2419]   8.3%  ok: 0  errori: 0
  --> Checkpoint salvato
  [ 210/2419]   8.7%  ok: 0  errori: 0
  [ 220/2419]   9.1% 